# Vídeo 1 – Demonstração Consolidada (OpenAI via .env)
Notebook consolidado com **Parte 1 (Problema: if/else hell)** e **Parte 2 (Solução: LangGraph com estado tipado)**.

Crie um arquivo `.env` com:
```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
```
Instale:
```
pip install -U python-dotenv openai langchain langchain-openai langgraph typing_extensions
```


## Parte 1 – Problema (procedural com LLM + APIs externas)

In [16]:
"""
Vídeo 1 – Introdução ao LangGraph
Arquivo: 01_problema_if_else_hell_openai.py

Objetivo: ilustrar o PROBLEMA do if/else hell quando misturamos LLM + regras + APIs externas.
Agora com chamadas reais à OpenAI via .env.

Dependências (instale antes):
    pip install python-dotenv openai langchain langchain-openai

Crie um arquivo .env ao lado deste script com:
    OPENAI_API_KEY=sk-...

Execução:
    python 01_problema_if_else_hell_openai.py
"""

import os
from typing import Dict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Carrega .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Defina OPENAI_API_KEY no arquivo .env.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128)

# ==============================
# Stubs de APIs externas
# ==============================

def consultar_fatura_api(cliente_id: str) -> Dict:
    return {"valor": 120.50, "vencimento": "10/09/2025"}

def registrar_cancelamento_api(cliente_id: str) -> bool:
    return True

def enviar_oferta_api(cliente_id: str, oferta: str) -> bool:
    return True

# ==============================
# Funções com LLM (procedural)
# ==============================

def classificar_intencao(mensagem: str) -> str:
    system = SystemMessage(content=(
        "Você é um classificador de intenção. Responda apenas com: faq, fatura ou cancelar."
    ))
    human = HumanMessage(content=f"Classifique: {mensagem}")
    out = llm.invoke([system, human]).content.strip().lower()
    if "fatura" in out:
        return "fatura"
    if "cancel" in out or "cancelar" in out:
        return "cancelar"
    return "faq"

def llm_responder_faq(mensagem: str) -> str:
    system = SystemMessage(content="Responda perguntas de FAQ de forma clara e breve.")
    return llm.invoke([system, HumanMessage(content=mensagem)]).content.strip()

def llm_resumir_fatura(dados: Dict) -> str:
    system = SystemMessage(content="Resuma dados de fatura em uma frase objetiva em português.")
    return llm.invoke([system, HumanMessage(content=str(dados))]).content.strip()

def llm_gerar_oferta() -> str:
    system = SystemMessage(content="Gere uma breve contra-oferta para retenção de cliente.")
    return llm.invoke([system, HumanMessage(content="Crie uma oferta atrativa")]).content.strip()

# ==============================
# Função principal (if/else hell)
# ==============================

def atender_usuario(mensagem: str, cliente: Dict) -> str:
    intencao = classificar_intencao(mensagem)

    if intencao == "faq":
        return llm_responder_faq(mensagem)

    elif intencao == "fatura":
        if not cliente.get("autenticado"):
            return "Usuário não autenticado. Solicitar login."
        dados = consultar_fatura_api(cliente["id"])
        resumo = llm_resumir_fatura(dados)
        return f"Resumo da fatura: {resumo}"

    elif intencao == "cancelar":
        if cliente.get("vip"):
            oferta = llm_gerar_oferta()
            enviar_oferta_api(cliente["id"], oferta)
            return f"Oferecer retenção: {oferta}"
        registrar_cancelamento_api(cliente["id"])
        return "Assinatura cancelada."

    return "Não entendi. Encaminhar para humano."

# ==============================
# Exemplos
# ==============================
if __name__ == "__main__":
    exemplos = [
        ("Quero ver a fatura deste mês", {"id": "123", "autenticado": True, "vip": False}),
        ("Quero cancelar minha assinatura", {"id": "321", "autenticado": True, "vip": True}),
        ("Qual o horário de atendimento?", {"id": "999", "autenticado": False, "vip": False}),
    ]

    print("=== Execução (PROBLEMA: if/else hell) ===\n")
    for msg, cli in exemplos:
        print(f"Usuário: {msg} | Cliente: {cli}")
        print("Resposta:", atender_usuario(msg, cli))
        print("-" * 60)


=== Execução (PROBLEMA: if/else hell) ===

Usuário: Quero ver a fatura deste mês | Cliente: {'id': '123', 'autenticado': True, 'vip': False}
Resposta: Resumo da fatura: Fatura no valor de R$ 120,50 com vencimento em 10 de setembro de 2025.
------------------------------------------------------------
Usuário: Quero cancelar minha assinatura | Cliente: {'id': '321', 'autenticado': True, 'vip': True}
Resposta: Oferecer retenção: Claro! Aqui está uma sugestão de contra-oferta para retenção de cliente:

---

**Prezado [Nome do Cliente],**

Agradecemos por ser parte da nossa família! Sabemos que você está considerando outras opções, e gostaríamos de apresentar uma oferta exclusiva para que continue conosco.

**Oferta Especial de Retenção:**

- **Desconto de 20%** na sua próxima renovação.
- **Acesso a um serviço premium** por 3 meses, sem custo adicional.
- **Suporte prioritário** para garantir que você tenha a melhor experiência possível.

Estamos comprometidos em oferecer o melhor para voc

In [17]:
# Executar exemplos da Parte 1 (procedural)
exemplos = [
    ("Quero ver a fatura deste mês", {"id": "123", "autenticado": True, "vip": False}),
    ("Quero cancelar minha assinatura", {"id": "321", "autenticado": True, "vip": True}),
    ("Qual o horário de atendimento?", {"id": "999", "autenticado": False, "vip": False}),
]
for msg, cli in exemplos:
    print('Usuário:', msg, '| Cliente:', cli)
    print('Resposta:', atender_usuario(msg, cli))
    print('-'*60)


Usuário: Quero ver a fatura deste mês | Cliente: {'id': '123', 'autenticado': True, 'vip': False}
Resposta: Resumo da fatura: Fatura no valor de R$ 120,50 com vencimento em 10 de setembro de 2025.
------------------------------------------------------------
Usuário: Quero cancelar minha assinatura | Cliente: {'id': '321', 'autenticado': True, 'vip': True}
Resposta: Oferecer retenção: Claro! Aqui está uma sugestão de contra-oferta para retenção de cliente:

---

**Prezado(a) [Nome do Cliente],**

Agradecemos por ser um cliente valioso para nós! Sabemos que você está considerando outras opções e, para demonstrar nosso compromisso em atendê-lo da melhor forma, gostaríamos de apresentar uma oferta exclusiva:

**Oferta Especial de Retenção:**

- **Desconto de 20%** na sua próxima renovação de contrato.
- **Acesso gratuito** a um serviço premium por 3 meses.
- **Suporte prioritário** com um gerente de conta dedicado para resolver suas
---------------------------------------------------------

## Parte 2 – Solução (LangGraph com estado tipado + agregador)

In [18]:
"""
Vídeo 1 – Solução com LangGraph (estado tipado + agregador)
Arquivo: 02_solucao_grafo_langgraph_openai_typed.py

Objetivo: habilitar o draw_ascii() real do LangGraph usando
TypedDict + Annotated com agregador (add_messages) para evitar
InvalidUpdateError em chaves que podem receber múltiplas escritas
no mesmo passo (ex.: 'historico').

Dependências:
    pip install python-dotenv openai langchain langchain-openai langgraph typing_extensions

.env (na mesma pasta):
    OPENAI_API_KEY=sk-...
    OPENAI_MODEL=gpt-4o-mini   # opcional

Execução:
    python 02_solucao_grafo_langgraph_openai_typed.py
"""

import os
from typing import Dict
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# ---------- OpenAI ----------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Defina OPENAI_API_KEY no arquivo .env.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128)

# ---------- Estado tipado ----------
class State(TypedDict, total=False):
    mensagem: str
    cliente: dict
    intencao: str
    fatura: dict
    oferta: str
    resposta: str
    erro: str
    # Campo com agregador: permite múltiplas escritas no mesmo passo sem conflito
    historico: Annotated[list, add_messages]

# ---------- APIs externas (stubs) ----------
def api_consultar_fatura(cliente_id: str) -> Dict:
    return {"valor": 120.50, "vencimento": "10/09/2025"}

def api_registrar_cancelamento(cliente_id: str) -> bool:
    return True

def api_enviar_oferta(cliente_id: str, oferta: str) -> bool:
    return True

# ---------- Nós do grafo ----------
def classificar_intencao(estado: State) -> State:
    mensagem = estado.get("mensagem", "")
    system = SystemMessage(content=(
        "Você é um classificador de intenção. Responda apenas com: faq, fatura ou cancelar."
    ))
    human = HumanMessage(content=f"Classifique: {mensagem}")
    out = llm.invoke([system, human]).content.strip().lower()
    if "fatura" in out:
        intent = "fatura"
    elif "cancel" in out or "cancelar" in out:
        intent = "cancelar"
    else:
        intent = "faq"
    return {
        "intencao": intent,
        "historico": [{"role": "system", "content": f"classificar_intencao -> {intent}"}],
    }

def responder_faq(estado: State) -> State:
    mensagem = estado.get("mensagem", "")
    system = SystemMessage(content="Responda perguntas de FAQ de forma clara e breve.")
    human = HumanMessage(content=mensagem)
    resposta = llm.invoke([system, human]).content.strip()
    return {
        "resposta": resposta,
        "historico": [{"role": "system", "content": "responder_faq"}],
    }

def consultar_fatura(estado: State) -> State:
    cliente = estado.get("cliente", {})
    if not cliente.get("autenticado"):
        return {
            "erro": "Usuário não autenticado. Solicitar login.",
            "historico": [{"role": "system", "content": "consultar_fatura -> erro_auth"}],
        }
    dados = api_consultar_fatura(cliente["id"])
    return {
        "fatura": dados,
        "historico": [{"role": "system", "content": "consultar_fatura"}],
    }

def resumir_fatura(estado: State) -> State:
    dados = estado.get("fatura", {})
    system = SystemMessage(content="Resuma dados de fatura em uma frase objetiva.")
    human = HumanMessage(content=f"Dados: {dados}")
    resumo = llm.invoke([system, human]).content.strip()
    return {
        "resposta": f"Resumo da fatura: {resumo}",
        "historico": [{"role": "system", "content": "resumir_fatura"}],
    }

def decidir_cancelamento(estado: State) -> State:
    # Nó de roteamento; ainda assim registra no histórico
    return {"historico": [{"role": "system", "content": "decidir_cancelamento"}]}

def gerar_oferta(estado: State) -> State:
    system = SystemMessage(content="Gere uma breve contra-oferta para retenção de cliente.")
    human = HumanMessage(content="Crie uma oferta atrativa e concisa.")
    oferta = llm.invoke([system, human]).content.strip()
    return {
        "oferta": oferta,
        "resposta": f"Oferecer retenção: {oferta}",
        "historico": [{"role": "system", "content": "gerar_oferta"}],
    }

def enviar_oferta(estado: State) -> State:
    cliente = estado.get("cliente", {})
    oferta = estado.get("oferta", "")
    api_enviar_oferta(cliente.get("id", ""), oferta)
    return {"historico": [{"role": "system", "content": "enviar_oferta"}]}

def registrar_cancelamento(estado: State) -> State:
    cliente = estado.get("cliente", {})
    api_registrar_cancelamento(cliente.get("id", ""))
    return {
        "resposta": "Assinatura cancelada.",
        "historico": [{"role": "system", "content": "registrar_cancelamento"}],
    }

# ---------- Construção do grafo ----------
grafo = StateGraph(State)

grafo.add_node("classificar", classificar_intencao)
grafo.add_node("responder_faq", responder_faq)
grafo.add_node("consultar_fatura", consultar_fatura)
grafo.add_node("resumir_fatura", resumir_fatura)
grafo.add_node("decidir_cancelamento", decidir_cancelamento)
grafo.add_node("gerar_oferta", gerar_oferta)
grafo.add_node("enviar_oferta", enviar_oferta)
grafo.add_node("registrar_cancelamento", registrar_cancelamento)

grafo.set_entry_point("classificar")

def rota_intencao(estado: State):
    return estado.get("intencao", "faq")

grafo.add_conditional_edges(
    "classificar",
    rota_intencao,
    {
        "faq": "responder_faq",
        "fatura": "consultar_fatura",
        "cancelar": "decidir_cancelamento",
    },
)

grafo.add_edge("consultar_fatura", "resumir_fatura")

def rota_cancelamento(estado: State):
    return "vip" if estado.get("cliente", {}).get("vip") else "normal"

grafo.add_conditional_edges(
    "decidir_cancelamento",
    rota_cancelamento,
    {
        "vip": "gerar_oferta",
        "normal": "registrar_cancelamento",
    },
)

grafo.add_edge("gerar_oferta", "enviar_oferta")
grafo.add_edge("resumir_fatura", END)
grafo.add_edge("responder_faq", END)
grafo.add_edge("enviar_oferta", END)
grafo.add_edge("registrar_cancelamento", END)

app = grafo.compile()

if __name__ == "__main__":
    print("=== Estrutura do grafo (ASCII - real) ===")
    print(app.get_graph().draw_ascii())

    print("\n=== Execução de exemplo (VIP) ===")
    estado_inicial = {
        "mensagem": "Quero cancelar minha assinatura",
        "cliente": {"id": "321", "autenticado": True, "vip": True},
    }
    print(app.invoke(estado_inicial))

    print("\n=== Execução de exemplo (Fatura) ===")
    estado_inicial = {
        "mensagem": "Quero ver a fatura",
        "cliente": {"id": "123", "autenticado": True, "vip": False},
    }
    print(app.invoke(estado_inicial))


=== Estrutura do grafo (ASCII - real) ===


ImportError: Install grandalf to draw graphs: `pip install grandalf`.

In [ ]:
# Mostrar o grafo (draw_ascii real) e executar exemplos da Parte 2
print(app.get_graph().draw_ascii())
print('\nExecução (VIP):')
print(app.invoke({
    'mensagem': 'Quero cancelar minha assinatura',
    'cliente': {'id': '321', 'autenticado': True, 'vip': True},
}))
print('\nExecução (Fatura):')
print(app.invoke({
    'mensagem': 'Quero ver a fatura',
    'cliente': {'id': '123', 'autenticado': True, 'vip': False},
}))
